# 01 - Exploratory Data Analysis (EDA)
**Traffic Sign Classifier + RL Policy Project**
This notebook explores the GTSRB dataset, analyzing class distributions, image dimensions, and sample visualizations. It fulfills the Week 2 EDA checkpoint deliverable for the Modeling Lead.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
import torch
from torchvision import datasets, transforms

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Load Dataset Information
We use the `torchvision` GTSRB dataset downloader to retrieve the raw images.

In [ ]:
# Load the raw train dataset without augmentations to analyze true data
data_root = "../data/"
train_dataset = datasets.GTSRB(root=data_root, split="train", download=True)

print(f"Total training images: {len(train_dataset)}")

## 2. Class Distribution Analysis
GTSRB has 43 distinct classes. Let's look at the balance of images per class. Class imbalance is a known challenge in traffic sign datasets.

In [ ]:
# Extract labels
labels = [label for _, label in train_dataset._samples]
label_counts = Counter(labels)

# Plot
plt.figure(figsize=(15, 6))
sns.barplot(x=list(label_counts.keys()), y=list(label_counts.values()), color='steelblue')
plt.title("GTSRB Training Set - Class Distribution", fontsize=16)
plt.xlabel("Class ID (0-42)", fontsize=12)
plt.ylabel("Number of Images", fontsize=12)
plt.xticks(rotation=90, fontsize=8)
plt.axhline(y=np.mean(list(label_counts.values())), color='r', linestyle='--', label='Mean Images/Class')
plt.legend()
plt.show()

print(f"Most common class: {label_counts.most_common(1)[0]}")
print(f"Least common class: {label_counts.most_common()[-1]}")

## 3. Image Dimensions Analysis
Unlike standard benchmark datasets (e.g., CIFAR-10), GTSRB images vary wildly in size and aspect ratio. We must analyze this to justify our `32x32` resize choice in the data pipeline.

In [ ]:
# Sample 2000 images to check dimensions
sample_indices = random.sample(range(len(train_dataset)), 2000)
widths, heights = [], []

for i in sample_indices:
    img_path = train_dataset._samples[i][0]
    with Image.open(img_path) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(widths, kde=True, ax=ax1, color='green')
ax1.set_title("Image Widths Distribution")
ax1.set_xlabel("Width (pixels)")

sns.histplot(heights, kde=True, ax=ax2, color='orange')
ax2.set_title("Image Heights Distribution")
ax2.set_xlabel("Height (pixels)")

plt.tight_layout()
plt.show()

print(f"Min size: {min(widths)}x{min(heights)}")
print(f"Max size: {max(widths)}x{max(heights)}")
print(f"Mean size: {int(np.mean(widths))}x{int(np.mean(heights))}")

## 4. Visualizing Sample Images
Let's visualize a grid of random traffic signs across different classes to understand lighting, blur, and occlusion challenges.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
sample_indices = random.sample(range(len(train_dataset)), 15)

for idx, ax in zip(sample_indices, axes.flatten()):
    img_path, label = train_dataset._samples[idx]
    with Image.open(img_path) as img:
        ax.imshow(img)
        ax.set_title(f"Class: {label}")
        ax.axis('off')

plt.suptitle("Random Samples from GTSRB Training Set", fontsize=16)
plt.tight_layout()
plt.show()

## 5. Conclusions for Modeling
1. **Class Imbalance**: Severe class imbalance exists (ranging from ~200 to ~2000 images per class). Our CNN training pipeline must account for this (e.g., via stratified splits, which we implemented in `src/data_pipeline.py`).
2. **Resolution Variance**: Images range from 25x25 up to 250x250. Standardizing inputs to `32x32` is a sound heuristic that balances detail preservation with computational efficiency.
3. **Artifacts**: Images suffer from poor lighting, physical damage, and motion blur. Data augmentation (color jitter, rotation) used in our pipeline is highly justified.